# Skills vs. Tools

One of the most common design mistakes in agent development is conflating two fundamentally different concepts: **skills** and **tools**. Both extend what an agent can do - but they extend it in completely different ways.

- A **tool** is a callable function the agent invokes to *perform an action* - read a file, search the web, run code.
- A **skill** is a knowledge and instruction package the agent *loads into context* - it shapes how the agent reasons, what procedures it follows, and which tools it is allowed to use for a category of work.

In [1]:
import os
from dataclasses import dataclass  # used later for the decision framework

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

### Initializing the LLMs

In [2]:
# A small, fast model is plenty for these illustrative examples
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)
print("Model ready:", llm.model)

Model ready: gpt-4o-mini


## The core distinction
It helps to have a precise mental model of what each concept *is*.

| Dimension | Tool | Skill |
|-----------|------|-------|
| **Nature** | Executable function | Knowledge + instruction package |
| **Lives in** | Tool registry / function definition | Context window (system prompt) |
| **Invocation** | Agent calls it at runtime | Loaded before task begins |
| **Output** | Data: text, JSON, numbers | Shaped agent behavior |
| **Scope** | Single operation | Entire task session |
| **Failure mode** | Throws error / returns null | Silent - poorly written skills just don't help |
| **Authoring** | Code (Python) | Markdown + YAML |
| **Testing** | Unit tests, mock returns | Behavioral evaluation, output consistency |

The key insight: tools *do things*; skills *know things*. A legal agent needs both - a tool to search case law databases, and a skill that knows *what* to search for, *why* it matters, and *how* to interpret what comes back.

Mixing these up causes real problems:
- **Domain logic in tools** makes tools brittle, non-reusable, and hard to update without redeployment
- **Executable logic in skills** makes skills unrunnable and agents that ignore them produce wrong results

## Building tools with LangChain
Tools are defined with the `@tool` decorator. The function docstring becomes the tool description the LLM uses to decide when and how to call it. Tools are then bound to the LLM with `llm.bind_tools()`.

Here we define three representative tools that an engineering agent might use. Notice that each tool is *generic* - it doesn't know anything about code review protocols, only how to perform a specific operation.

In [3]:
@tool
def read_file(path: str) -> str:
    """Read a file from disk and return its contents as a string."""
    # Simulated file content for demonstration
    mock_files = {
        "discount.py": """\
def calculate_discount(price, discount_pct):
    return price - (price * discount_pct)

# Usage
final = calculate_discount(100, 20)  # Expected: 80.0, Actual: -1900.0
print(f"Discounted price: {final}")
""",
        "auth.py": """\
def login(username, password):
    query = f"SELECT * FROM users WHERE name='{username}' AND pass='{password}'"
    return db.execute(query)
""",
    }
    return mock_files.get(path, f"File not found: {path}")


@tool
def run_linter(code: str) -> str:
    """Run a static linter on Python code and return a list of style and type issues."""
    # Simulated linter output
    issues = []
    if "discount_pct" in code and "* discount_pct" in code:
        issues.append("Line 2: discount_pct appears to be a percentage (20) but is used as a fraction — should be discount_pct / 100")
    if "f\"SELECT" in code or "f'SELECT" in code:
        issues.append("Line 2: SQL query built with f-string — potential SQL injection vulnerability")
    if not issues:
        issues.append("No linter issues found")
    return "\n".join(issues)


@tool
def search_web(query: str) -> str:
    """Search the web and return a brief summary of the top results."""
    # Simulated search results
    return f"Search results for '{query}': Found documentation on Python percentage calculations and best practices for numeric validation."


tools = [read_file, run_linter, search_web]
print("Tools registered:")
for t in tools:
    print(f"  {t.name}: {t.description[:60]}...")

Tools registered:
  read_file: Read a file from disk and return its contents as a string....
  run_linter: Run a static linter on Python code and return a list of styl...
  search_web: Search the web and return a brief summary of the top results...


Technically, the cell above establishes the *action layer* of the agent:

- **`@tool` decorator** - converts each plain Python function into a LangChain tool, deriving the tool name from the function name and the tool *description* from the docstring. That description is the only thing the LLM sees when deciding whether to call the function.
- **Type hints as a schema** - the `path: str` / `code: str` annotations are compiled into the JSON schema the model uses to format its arguments.
- **Mocked side effects** - `read_file`, `run_linter`, and `search_web` return canned data so the notebook runs offline; in production these would touch a real filesystem, linter process, or search API.
- **Generic by design** - none of the three functions contains any code-review judgment. They retrieve or compute; deciding *what the results mean* is left to the caller. That separation is exactly the point this notebook builds toward.

Now we bind the tools to the LLM and run a code review task. Watch what happens when the agent has only tools and no skill guidance - it can *access* data but applies only generic reasoning.

In [4]:
# Bind tools to the LLM — the model can now request tool calls
llm_with_tools = llm.bind_tools(tools)
tools_by_name = {t.name: t for t in tools}

def run_tool_agent(user_message: str, max_steps: int = 4) -> str:
    """Simple ReAct loop: LLM reasons, calls tools, loops until done."""
    messages = [HumanMessage(content=user_message)]

    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If no tool calls, agent is done
        if not response.tool_calls:
            return response.content

        # Execute each tool call and feed results back
        for call in response.tool_calls:
            tool_fn = tools_by_name[call["name"]]
            result = tool_fn.invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
            print(f"  [tool] {call['name']}({call['args']}) → {str(result)[:80]}...")

    return messages[-1].content


print("=" * 55)
print("TOOLS ONLY (no skill) — code review")
print("=" * 55)
result_tools_only = run_tool_agent(
    "Review the file discount.py for any bugs or issues."
)
print("\nAgent output:")
print(result_tools_only)

TOOLS ONLY (no skill) — code review
  [tool] read_file({'path': 'discount.py'}) → def calculate_discount(price, discount_pct):
    return price - (price * discoun...
  [tool] run_linter({'code': 'def calculate_discount(price, discount_pct):\n    return price - (price * discount_pct)\n\n# Usage\nfinal = calculate_discount(100, 20)  # Expected: 80.0, Actual: -1900.0\nprint(f"Discounted price: {final}")'}) → Line 2: discount_pct appears to be a percentage (20) but is used as a fraction —...

Agent output:
The review of the `discount.py` file revealed the following issues:

1. **Logical Error in Discount Calculation**:
   - The `discount_pct` is expected to be a percentage (e.g., 20 for 20%), but it is used directly in the calculation without converting it to a fraction. This leads to incorrect results. The calculation should be modified to divide `discount_pct` by 100.

2. **Linter Warning**:
   - The linter pointed out that `discount_pct` appears to be a percentage but is used as a fract

The agent found *something*, but the output format is inconsistent and the reasoning is ad hoc. There's no severity classification, no structured report, no strengths section - just whatever the model felt like saying. Each time we run this, the structure and emphasis may differ.

This is the fundamental limitation of tools-without-skills: tools provide **data**, not **judgment** about how to apply that data.

## Building a skill
A skill is defined in a `SKILL.md` file - YAML frontmatter for metadata plus a Markdown instruction body. In a running agent, the skill body is injected as the system prompt before the task begins.

The instruction body encodes:
- **When** to apply this skill (trigger conditions)
- **How** to perform the work (step-by-step workflow)
- **What** constraints to follow (format, tool access, escalation)

Below we define a `code-review` skill and run the same task. No tools - just skill-guided reasoning.

In [5]:
# This represents the body of a SKILL.md file loaded into context
CODE_REVIEW_SKILL = """
You have the Code Review skill active.

## When to use this skill
Activate whenever asked to review, audit, or assess a code change, pull request,
diff, or code snippet for quality, correctness, or security.

## Review workflow
1. Read the entire code before forming any judgment — do not comment mid-read
2. Classify every issue by severity:
   - CRITICAL: bugs that cause wrong results, crashes, or data loss (must fix)
   - MAJOR: logic errors, missing validation, security vulnerabilities (should fix)
   - MINOR: style, naming, documentation, readability (nice to fix)
3. For each issue state: the line number, the problem, and a specific fix
4. Include a Strengths section — what the code does well
5. End with one-sentence OVERALL verdict

## Output format (always follow exactly)
CRITICAL ISSUES:
  1. Line X — <problem> → Fix: <specific fix>

MAJOR ISSUES:
  1. Line X — <problem> → Fix: <specific fix>

MINOR ISSUES:
  1. Line X — <problem> → Fix: <specific fix>

STRENGTHS:
  - <bullet points>

OVERALL: <one sentence verdict>
"""

# Sample code to review — same as before
SAMPLE_CODE = """\
def calculate_discount(price, discount_pct):
    return price - (price * discount_pct)

# Usage
final = calculate_discount(100, 20)  # Expected: 80.0, Actual: -1900.0
print(f"Discounted price: {final}")
"""

# Inject skill as system context — this is the core mechanism
messages = [
    SystemMessage(content=CODE_REVIEW_SKILL),
    HumanMessage(content=f"Review this code:\n\n```python\n{SAMPLE_CODE}\n```"),
]

response = llm.invoke(messages)

print("=" * 55)
print("SKILL ONLY (no tools) — code review")
print("=" * 55)
print(response.content)

SKILL ONLY (no tools) — code review
CRITICAL ISSUES:
  1. Line 2 — The discount calculation is incorrect, leading to wrong results → Fix: Change the calculation to `return price - (price * (discount_pct / 100))` to correctly apply the percentage discount.

MAJOR ISSUES:
  1. Line 1 — Missing validation for input parameters (e.g., negative price or discount percentage) → Fix: Add validation checks to ensure `price` is non-negative and `discount_pct` is between 0 and 100.

MINOR ISSUES:
  1. Line 1 — Function lacks a docstring explaining its purpose and parameters → Fix: Add a docstring to describe the function's behavior and its parameters.

STRENGTHS:
  - The function is straightforward and uses a simple formula for discount calculation.
  - The usage example is clear and demonstrates how to call the function.

OVERALL: The code needs critical fixes to ensure correct discount calculations and input validation.


The output is now structured, consistent, and expert-level - even without calling any tools. The skill encoded the review protocol: severity tiers, output format, strengths section, and overall verdict. Run this cell multiple times and we will get the same structure every time, because the skill defines the behavioral contract.

This is what skills provide: **repeatable, consistent expertise**.

## Skills guiding tools - the production pattern
In real agents, skills and tools work together. The skill defines *what to do and in what order*; tools provide the data and actions needed to do it. The skill also narrows which tools the agent may use - via the `allowed_tools` field in the SKILL.md frontmatter.

Here we combine both: the code review skill in the system prompt plus tools bound to the LLM. The skill instructs the agent to use `read_file` and `run_linter`; the agent follows this guidance.

In [6]:
# A SKILL.md file opens with a YAML frontmatter block; here we model that block as a Python dict.
# NOTE: in a real SKILL.md the tool-restriction field is written `allowed-tools` (hyphenated, per the Agent Skills spec).
# We use the underscore form `allowed_tools` so the same key is a valid Python identifier throughout.
SKILL_FRONTMATTER = {
    "name": "code-review",
    "description": "Reviews code changes for correctness, security, and maintainability",
    "version": "1.0.0",
    "allowed_tools": ["read_file", "run_linter"],  # search_web deliberately NOT permitted
}

def scope_tools_to_skill(all_tools: list, allowed_names: list) -> list:
    """Filter the full tool registry down to only the tools the active skill permits."""
    return [t for t in all_tools if t.name in allowed_names]


# Expose only the tools this skill declares — the skill narrows the agent's reach
skill_tools = scope_tools_to_skill(tools, SKILL_FRONTMATTER["allowed_tools"])
print(f"Full tool registry:        {[t.name for t in tools]}")
print(f"Tools permitted by skill:  {[t.name for t in skill_tools]}")
print(f"Tool scoping removes:      {set(t.name for t in tools) - set(t.name for t in skill_tools)}")

Full tool registry:        ['read_file', 'run_linter', 'search_web']
Tools permitted by skill:  ['read_file', 'run_linter']
Tool scoping removes:      {'search_web'}


What just happened, technically:
- **`SKILL_FRONTMATTER`** stands in for the YAML header of a `SKILL.md` file. The `allowed_tools` list is the security-relevant part - it enumerates exactly which tools this skill is trusted to drive.
- **`scope_tools_to_skill`** intersects the global registry with that allow-list, returning a narrowed tool set. `search_web` is filtered out: there is no reason a code review should reach out to the open web, so the skill removes the capability rather than relying on the model to abstain.

With the tool set narrowed, the next step wires the two halves together - the skill body goes into the system prompt for *judgment*, and the scoped tools are bound to the LLM for *data access*. The agent loop below drives both. Note that it is parameterized by its tool list, so we can reuse the exact same loop later for a different skill.

In [7]:
def run_skill_guided_agent(user_message: str, skill_body: str, agent_tools: list, max_steps: int = 5) -> str:
    """Run an agent loop with skill context in the system prompt and a scoped tool set.

    Args:
        user_message: The task for the agent.
        skill_body: The SKILL.md instruction body, injected as the system prompt.
        agent_tools: The tools the skill permits — bound to the LLM for this run only.
        max_steps: Safety cap on reasoning/tool iterations.
    """
    # Bind only the scoped tools for this run, and index them by name for execution
    bound_llm = llm.bind_tools(agent_tools)
    tools_by_name = {t.name: t for t in agent_tools}

    # Skill body becomes the system prompt (the "how-to"); user message is the task
    messages = [
        SystemMessage(content=skill_body),
        HumanMessage(content=user_message),
    ]

    for step in range(max_steps):
        response = bound_llm.invoke(messages)
        messages.append(response)

        # No tool calls means the model produced its final answer
        if not response.tool_calls:
            return response.content

        # Otherwise execute each requested tool and feed the result back in
        for call in response.tool_calls:
            if call["name"] not in tools_by_name:
                # Defense in depth: even if the model invents a tool, scoping blocks it
                result = f"ERROR: Tool '{call['name']}' is not permitted by the active skill."
            else:
                result = tools_by_name[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
            print(f"  [tool] {call['name']}({call['args']}) → {str(result)[:80]}")

    return messages[-1].content


print("=" * 55)
print("SKILL + TOOLS — code review (production pattern)")
print("=" * 55)
# Pass the scoped tool set so the agent can read the file and lint it, but not search the web
result_combined = run_skill_guided_agent(
    "Please review the file discount.py",
    CODE_REVIEW_SKILL,
    skill_tools,
)
print("\nAgent output:")
print(result_combined)

SKILL + TOOLS — code review (production pattern)
  [tool] read_file({'path': 'discount.py'}) → def calculate_discount(price, discount_pct):
    return price - (price * discoun
  [tool] run_linter({'code': 'def calculate_discount(price, discount_pct):\n    return price - (price * discount_pct)\n\n# Usage\nfinal = calculate_discount(100, 20)  # Expected: 80.0, Actual: -1900.0\nprint(f"Discounted price: {final}")\n'}) → Line 2: discount_pct appears to be a percentage (20) but is used as a fraction —

Agent output:
CRITICAL ISSUES:
  1. Line 2 — The discount percentage is incorrectly applied as a whole number instead of a fraction → Fix: Change the calculation to `return price - (price * (discount_pct / 100))`.

MAJOR ISSUES:
  1. Line 5 — The comment indicates an expected result that is incorrect based on the current implementation → Fix: Update the comment to reflect the correct expected result after applying the discount.

MINOR ISSUES:
  1. Line 1 — The function lacks a docstring expla

This is the production pattern in action:
- The **skill** provides the review protocol - the *how-to* expertise
- The **tools** provide the data - file contents and linter output
- Tool scoping ensures the agent can only call `read_file` and `run_linter` during a review task - `search_web` is blocked even though it's available in the registry

Neither skills alone nor tools alone produce this result. Together, they create an agent that acts with domain expertise *and* has access to real data.

Use this table as a quick reference when designing agents:

| You need to... | Use |
|----------------|-----|
| Read a file, call an API, run a query | **Tool** |
| Apply domain expertise consistently | **Skill** |
| Follow the same multi-step workflow every time | **Skill** |
| Produce output in a specific format reliably | **Skill** |
| Perform an action AND apply expert judgment | **Skill + Tool** |
| Make tools available but restrict which ones a task can use | **Skill** (`allowed_tools`) |
| Encode company-specific procedures | **Skill** |
| Compute or retrieve data generically | **Tool** |

- **Tools execute; skills prepare.** Tools are Python functions the agent calls. Skills are knowledge packages loaded into the agent's context before planning begins.
- **Skills and tools are complementary.** In production agents, skills guide *which* tools to use and *how to interpret* results. Neither is sufficient alone for complex domain tasks.
- **Domain logic belongs in skills, not tools.** Generic tools (read_file, extract_patterns) are reusable and maintainable. Domain judgment lives in skill Markdown - editable without redeployment.
- **Use both when a task requires external data AND domain expertise.** The skill defines the protocol; the tool provides the data; the LLM synthesizes both.